In [1]:
import sys
import os
import json
import random
import matplotlib.pyplot as plt
from IPython.display import clear_output
import numpy as np
import pandas as pd
import geopandas as gpd
import shutil
import os
    

sys.path.append('../')
import model.dataloader as dataloader
from model.dataset import LABEL_MAPPING


In [2]:
inverse_label_mapping = {value : key for key, value in LABEL_MAPPING.items()}

In [3]:
_id = 'aew9h7hg'


In [4]:


model_dir = os.path.join('../models_trained', _id)
config_path = os.path.join(model_dir, 'config.json')
with open(config_path) as source:
    config = json.loads(source.read())


In [6]:
config['data']['annotation_file'] =  '/home/jan/Documents/EPFL/project_2/data/labels/dataset_v5/labels_no_regrowth_fixed_country_names.csv'
config['hyperparams']['image_encoder']['use'] = False

In [8]:
# fetch dataloaders
dataloaders = dataloader.fetch_dataloader(
    ['train', 'val'], config)
train_dl = dataloaders['train']
val_dl = dataloaders['val']

In [9]:
x,y = next(iter(train_dl))

In [10]:
x[1].shape

torch.Size([64, 2])

In [11]:
train_dl.dataset.img_labels.iloc[0]

driver                                              cocoa
sampleid                                        170000012
dataset                             cocoa_peru_points.csv
country                                              Peru
geometry    POINT (-75.86782891784124 -6.463607769725906)
block_id                                             67.0
split                                               train
dst_size                                            100.0
Name: 0, dtype: object

In [14]:
device = 'cuda'

In [42]:
all_rates = []
for train_batch, labels_batch in train_dl:

    arr = train_batch[2].numpy()
    #print(arr.shape)
    all_rates.append(arr)
    

In [43]:
train_tab_values = np.concatenate(all_rates)

In [44]:
all_rates = []
for train_batch, labels_batch in val_dl:

    arr = train_batch[2].numpy()
    #print(arr.shape)
    all_rates.append(arr)
    

In [45]:
test_tab_values = np.concatenate(all_rates)

In [46]:
print(train_tab_values.shape)

(10000, 11)


In [48]:
tab_values = np.concatenate([train_tab_values, test_tab_values])

In [49]:
tab_values.shape

(14755, 11)

In [58]:
df = pd.read_csv('../data/ZENODO/labels.csv')
train = df.loc[df.dst_size <= 10000]
test = df.loc[df.split == 4]
df = pd.concat([train, test]).reset_index(drop=True)


In [60]:
LABEL_MAPPING = {
    'pasture' : 0, 
    'forest plantation' : 1,
    'soy' : 2,
    'coffee' : 3,
    'palm oil': 4,
    'mining': 5,
    'urban': 6,
    'cocoa': 7,
    'rubber': 8,
    'shifting agriculture': 9,
    'other': 10 
}

classes = [key for key in LABEL_MAPPING.keys()]

In [67]:
df[classes] = tab_values

In [71]:
df = df.drop(['dataset', 'block_id', 'dst_size'], axis=1)

In [79]:
df['split'] = df['split'].apply(lambda x: 'test' if x == 4 else 'train')

In [82]:
df.to_csv('../data/ZENODO/labels.csv', index=False)

## Copy images and deforestation masks

In [15]:
df = pd.read_csv('../data/ZENODO/labels.csv')

In [3]:

sampleids = df.sampleid.tolist()

In [4]:
import tqdm

In [6]:

img_dir = '/home/jan/Documents/EPFL/project_2/data/images/dataset_v5/four_images/'
forestloss_dir = '/home/jan/Documents/EPFL/project_2/data/forestloss_tiles/dataset_v5/'
img_target_dir = '/home/jan/Documents/EPFL/project_2/data/ZENODO/images/'
forestloss_target_dir = '/home/jan/Documents/EPFL/project_2/data/ZENODO/deforestation_masks/'
for sampleid in tqdm.tqdm(sampleids):
    shutil.copytree(os.path.join(img_dir, str(sampleid)), os.path.join(img_target_dir, str(sampleid)))
    shutil.copy(os.path.join(forestloss_dir, str(sampleid) + ".tif"), forestloss_target_dir)
    

100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 14755/14755 [03:32<00:00, 69.37it/s]


In [7]:
for sampleid in tqdm.tqdm(sampleids):
    assert len(os.listdir(os.path.join(img_target_dir, str(sampleid)))) == 4

100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 14755/14755 [00:00<00:00, 65218.93it/s]


In [9]:
df.columns

Index(['driver', 'sampleid', 'country', 'geometry', 'split', 'pasture',
       'forest plantation', 'soy', 'coffee', 'palm oil', 'mining', 'urban',
       'cocoa', 'rubber', 'shifting agriculture', 'other'],
      dtype='object')

In [16]:
tab_data = df[['sampleid', 'pasture',
       'forest plantation', 'soy', 'coffee', 'palm oil', 'mining', 'urban',
       'cocoa', 'rubber', 'shifting agriculture', 'other']]

In [17]:
df = df[['driver', 'sampleid', 'country', 'geometry', 'split']]

In [20]:
tab_data.to_csv('../data/ZENODO/tabular_data.csv', index=False)

In [21]:
df.to_csv('../data/ZENODO/labels.csv', index=False)

In [23]:
tab_data.columns

Index(['sampleid', 'pasture', 'forest plantation', 'soy', 'coffee', 'palm oil',
       'mining', 'urban', 'cocoa', 'rubber', 'shifting agriculture', 'other'],
      dtype='object')